# Reinforcement Learning From AI Feedback (RLAIF) with Serverless customization on SageMaker AI

## Lab 4.1 - Data preparation

In this notebook, we are going to prepare the dataset for later for fine-tuning Qwen 2.5 - 7B Instruct

---

## Prerequisites
### Install requirements

In [ ]:
%pip install -r requirements.txt

### AWS Access Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

project_prefix = "humanlike-rlaif"

---

##  Prepare the dataset

We are going to use the [Human-Like-DPO-Dataset](https://huggingface.co/datasets/HumanLLMs/Human-Like-DPO-Dataset) that was created as part of research aimed at improving conversational fluency and engagement in large language models. It is suitable for formats like Direct Preference Optimization (DPO) to guide models toward generating more human-like responses, although in this example we'll use just the chosen response to implement RLAIF.

### Download and preview the dataset

In [ ]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train")

df = pd.DataFrame(dataset)
df.head()

### Split and transform the dataset

For this workshop, we only require a subset of the dataset. For the evaluation dataset we limit the number of samples to 100 to make sure the lab completes in a reasonable time. The maximum supported number of samples by LLM as a Judge (LLMaaJ) is 1000.

In [ ]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(df, train_size=8000, test_size=100, random_state=42)

print("Number of train elements: ", len(train))
print("Number of test elements: ", len(val))

Next, we format the datasets for training and evaluation and save them to JSONL files. The training dataset needs to follow the [VERL post-training format](https://verl.readthedocs.io/en/latest/preparation/prepare_data.html) which is required by the SageMaker SDK [RLAIFTrainer](https://sagemaker.readthedocs.io/en/stable/v3-examples/model-customization-examples/rlaif_finetuning_example_notebook_v3_prod.html).

In [ ]:
def prepare_dataset_sm_rlaif(sample):
    try:
        return {
            "data_source": "HumanLLMs/Human-Like-DPO-Dataset",
            "prompt": [
                {
                    "role": "user",
                    "content": sample.get("prompt")
                }
            ],
            "ability": "human-like",
            "reward_model": {
                "ground_truth": sample.get("chosen"),
                "style": "llmj",
            }
        }
    except Exception as e:
        print(f"Error: {e}")
        raise e
    
def prepare_dataset_sm_eval(sample):
    try:
        return {
            "query": sample.get("prompt"),
            "response": sample.get("chosen")
        }
    except Exception as e:
        print(f"Error: {e}")

        raise e

In [ ]:
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(train)

val_dataset = Dataset.from_pandas(val)

dataset = DatasetDict({"train": train_dataset, "eval": val_dataset})

train_dataset = dataset["train"].map(
    prepare_dataset_sm_rlaif, remove_columns=list(train_dataset.features)
)

val_dataset = dataset["eval"].map(
    prepare_dataset_sm_eval, remove_columns=list(val_dataset.features)
)

train_dataset.to_json(f"./datasets/{project_prefix}_train.jsonl", orient="records")
val_dataset.to_json(f"./datasets/{project_prefix}_eval.jsonl", orient="records")

Let's show an example entry of the final, transformed dataset.

In [ ]:
from rich.pretty import pprint

# View a single sample
pprint(train_dataset[0])

### Upload the datasets to S3

In [ ]:
import os

bucket_name = sess.default_bucket()

local_path = "./datasets/"
file_name = f"{project_prefix}_train.jsonl"
prefix_key = f"{project_prefix}/{file_name}"
s3_client.upload_file(os.path.join(local_path, file_name), bucket_name, prefix_key)

train_data_uri = f"s3://{bucket_name}/{prefix_key}"
print(train_data_uri)

file_name = f"{project_prefix}_eval.jsonl"
prefix_key = f"{project_prefix}/{file_name}"
s3_client.upload_file(os.path.join(local_path, file_name), bucket_name, prefix_key)

eval_data_uri = f"s3://{bucket_name}/{prefix_key}"
print(eval_data_uri)

### Register the datasets in the SageMaker AI registry

Lastly, we register the datasets as reusable assets in the SageMaker AI registry, which will make them available for us in the subsequent notebooks for configuring the training and evaluation jobs.

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

# Register datasets in SageMaker AI Registry.
train_dataset = DataSet.create(
    name=f"{project_prefix}-train",
    source=train_data_uri,
    sagemaker_session=sess,
    wait=True
)

print(f"Training dataset ARN: {train_dataset.arn}")
train_dataset_arn = train_dataset.arn

eval_dataset = DataSet.create(
    name=f"{project_prefix}-eval",
    source=eval_data_uri,
    sagemaker_session=sess,
    wait=True
)

print(f"Evaluation dataset ARN: {eval_dataset.arn}")
eval_dataset_arn = eval_dataset.arn